[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andreas-he/ai-safety-challenges/blob/main/challenges/2026-04-30-logit-lens/challenge.ipynb)

# Logit Lens — Watching Predictions Form Layer by Layer

**Category:** Mech Interp &nbsp;|&nbsp; **Format:** short-coding &nbsp;|&nbsp; **Difficulty:** standard

**Why this matters for safety:** The logit lens reveals *when* a transformer commits to a prediction and *which layers* are responsible for building that commitment. For the SAIGE inoculation project (where you need to detect split/hidden model behaviours) and LASR Stage 3 (capability evaluation), understanding layer-wise representational dynamics is essential: a model that sandbags or behaves differently in evaluation contexts may show anomalous early-layer confidence patterns detectable with exactly this tool.

## Problem Summary
A transformer's residual stream at each layer encodes a "current best guess" about the next token. The **logit lens** makes this explicit: project each layer's residual stream through the unembedding matrix and take a softmax. You will implement three functions:
1. `logit_lens` — project all layer residuals through the unembedding matrix
2. `token_confidence_curve` — extract per-layer softmax probability for a specific target token
3. `first_confident_layer` — find the earliest layer exceeding a confidence threshold

## Setup

In [ ]:
import numpy as np

np.random.seed(42)

# Shared test fixtures — do not modify
N_LAYERS, SEQ_LEN, D_MODEL, D_VOCAB = 4, 3, 8, 10
residuals = np.random.randn(N_LAYERS, SEQ_LEN, D_MODEL)  # [layers, positions, d_model]
W_U = np.random.randn(D_MODEL, D_VOCAB)                  # [d_model, d_vocab]

def softmax(x, axis=-1):
    """Numerically stable softmax."""
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

print(f'residuals: {residuals.shape}, W_U: {W_U.shape}')

## Task 1 — `logit_lens(residuals, W_U)`

Apply the unembedding matrix to **every layer's** residual stream in one batched operation.

- **Input:** `residuals` of shape `[n_layers, seq_len, d_model]` and `W_U` of shape `[d_model, d_vocab]`
- **Output:** logits of shape `[n_layers, seq_len, d_vocab]`

The transformer's final-layer output is just a special case of this: `logit_lens(residuals, W_U)[-1]` equals the model's normal logits. The power of the logit lens is that *all* layers are available simultaneously.

*Hint: a single `@` (matmul) on numpy arrays handles batch dimensions correctly.*

In [ ]:
def logit_lens(residuals, W_U):
    """
    Project all layer residuals through the unembedding matrix.

    Args:
        residuals: np.ndarray, shape [n_layers, seq_len, d_model]
        W_U:       np.ndarray, shape [d_model, d_vocab]

    Returns:
        np.ndarray, shape [n_layers, seq_len, d_vocab]
    """
    raise NotImplementedError

### Tests — Task 1

In [ ]:
lens_out = logit_lens(residuals, W_U)

assert lens_out.shape == (N_LAYERS, SEQ_LEN, D_VOCAB), \
    f'Expected shape {(N_LAYERS, SEQ_LEN, D_VOCAB)}, got {lens_out.shape}'

# Layer 0, position 0 must equal the plain matmul
assert np.allclose(lens_out[0, 0], residuals[0, 0] @ W_U, atol=1e-6), \
    'Layer 0 output does not match expected matrix multiply'

# Last layer must match final-layer logits
assert np.allclose(lens_out[-1], residuals[-1] @ W_U, atol=1e-6), \
    'Last layer output mismatch'

print('Task 1 passed ✓')

## Task 2 — `token_confidence_curve(logit_lens_out, target_token_idx)`

Given the full logit-lens output, extract how the model's **confidence in a specific token** evolves across layers and positions.

- **Input:** `logit_lens_out` of shape `[n_layers, seq_len, d_vocab]` and a scalar `target_token_idx`
- **Output:** probability array of shape `[n_layers, seq_len]`

Apply softmax over the vocab dimension for each (layer, position) pair, then index into the target token. This curve is the core diagnostic: a model that "knows the answer" from early layers will show high confidence from layer 0; a model that needs deeper computation will show low confidence until a specific layer.

In [ ]:
def token_confidence_curve(logit_lens_out, target_token_idx):
    """
    Per-layer probability of the target token.

    Args:
        logit_lens_out:   np.ndarray, shape [n_layers, seq_len, d_vocab]
        target_token_idx: int

    Returns:
        np.ndarray, shape [n_layers, seq_len]
    """
    raise NotImplementedError

### Tests — Task 2

In [ ]:
TARGET = 3
conf = token_confidence_curve(lens_out, TARGET)

assert conf.shape == (N_LAYERS, SEQ_LEN), \
    f'Expected shape {(N_LAYERS, SEQ_LEN)}, got {conf.shape}'

assert np.all(conf >= 0) and np.all(conf <= 1), \
    'Probabilities must be in [0, 1]'

# Verify against manual softmax at layer 0, position 0
expected_l0_p0 = softmax(lens_out[0, 0])[TARGET]
assert np.isclose(conf[0, 0], expected_l0_p0, atol=1e-6), \
    f'Expected {expected_l0_p0:.6f}, got {conf[0, 0]:.6f}'

# Sanity: probs should sum to roughly 1 over vocab (only TARGET slice, so this is not 1 — skip)
print('Task 2 passed ✓')
print(f'Confidence of token {TARGET} at final layer, position 0: {conf[-1, 0]:.4f}')

## Task 3 — `first_confident_layer(confidence_curve, threshold=0.5)`

For each sequence position, find the **earliest layer** at which the model's confidence for the target token first exceeds `threshold`. If no layer ever reaches the threshold, return `-1` for that position.

- **Input:** `confidence_curve` of shape `[n_layers, seq_len]` and a float `threshold`
- **Output:** integer array of shape `[seq_len]` — the first layer index >= threshold, or -1

This is the "commitment point" of the model. In safety research, anomalously early commitment (confident at layer 1 of 40) can signal that a model has memorised or short-circuited the computation rather than reasoning properly — a potential indicator of deceptive or brittle behaviour.

In [ ]:
def first_confident_layer(confidence_curve, threshold=0.5):
    """
    Earliest layer exceeding the confidence threshold for each position.

    Args:
        confidence_curve: np.ndarray, shape [n_layers, seq_len]
        threshold:        float

    Returns:
        np.ndarray of int, shape [seq_len]
        Value is the first layer index where confidence >= threshold, or -1 if never.
    """
    raise NotImplementedError

### Tests — Task 3

In [ ]:
# Handcrafted curve: 3 positions, 4 layers
test_curve = np.array([
    [0.1, 0.8, 0.0],   # layer 0
    [0.3, 0.9, 0.0],   # layer 1
    [0.6, 0.95, 0.7],  # layer 2
    [0.9, 0.99, 0.8],  # layer 3
])

result = first_confident_layer(test_curve, threshold=0.5)
assert result.tolist() == [2, 0, 2], \
    f'Expected [2, 0, 2], got {result.tolist()}'

# No position ever reaches threshold 0.99
result2 = first_confident_layer(test_curve, threshold=0.99)
assert result2.tolist() == [-1, 0, -1], \
    f'Expected [-1, 0, -1], got {result2.tolist()}'

# All -1 when threshold is impossible
never = np.array([[0.1, 0.2], [0.3, 0.4]])
result3 = first_confident_layer(never, threshold=0.9)
assert result3.tolist() == [-1, -1], \
    f'Expected [-1, -1], got {result3.tolist()}'

print('Task 3 passed ✓')
print(f'First-confident-layer for each position (threshold=0.5): {result.tolist()}')

## Reflection

The logit lens exposes a single core fact about transformers: the residual stream is not just a communication bus — it is a *running prediction* that is iteratively refined. This has direct implications for mechanistic interpretability: circuits that "write" to the residual stream are circuits that *change the model's prediction*, and the logit lens makes that change directly observable.

> 📚 **Further reading:** Neel Nanda et al. (Google DeepMind, 2025), *A Pragmatic Vision for Interpretability* — https://www.lesswrong.com/posts/StENzDcD3kpfGJssR/a-pragmatic-vision-for-interpretability — discusses how tools like the logit lens fit into a broader interpretability methodology for safety-relevant work.

<details>
<summary><b>Solution</b></summary>

```python
def logit_lens(residuals, W_U):
    return residuals @ W_U

def token_confidence_curve(logit_lens_out, target_token_idx):
    probs = softmax(logit_lens_out)           # [n_layers, seq_len, d_vocab]
    return probs[:, :, target_token_idx]      # [n_layers, seq_len]

def first_confident_layer(confidence_curve, threshold=0.5):
    above = confidence_curve >= threshold     # [n_layers, seq_len]
    has_any = above.any(axis=0)               # [seq_len]
    return np.where(has_any, np.argmax(above, axis=0), -1)

# Key insight:
# logit_lens is just residuals @ W_U — a single matmul handles all layers via broadcasting.
# token_confidence_curve applies softmax then slices the target column.
# first_confident_layer uses argmax on a boolean mask (True > False in numpy),
#   with np.where to substitute -1 for positions that never cross the threshold.
# np.argmax returns 0 when no True is present, so the np.where guard is essential.
```

**Key insight — Task 1:** `residuals @ W_U` batches over all (layer, position) pairs in one call. No loop needed. Numpy broadcasts `[n_layers, seq_len, d_model] @ [d_model, d_vocab]` correctly.

**Key insight — Task 2:** Apply `softmax` over `axis=-1` (the vocab dimension) to get a proper probability distribution, then index `[:, :, target_token_idx]` to extract the target column. Avoid computing softmax token-by-token in a loop — it's 4× slower and produces identical results.

**Key insight — Task 3:** `np.argmax` on a boolean array returns the *first* `True` index (layer). But `argmax` returns 0 when no element is True — indistinguishable from "layer 0 was confident". The `np.where(has_any, argmax_result, -1)` pattern is the standard guard.

</details>